In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import col, current_timestamp, row_number
from pyspark.sql.window import Window


In [0]:
catalog_name = "charles_schwab_retailbrokerage_dev_team_lemma"
bronze_table = f"{catalog_name}.bronze.batchdate"
silver_table = f"{catalog_name}.silver.batchdate"

In [0]:

# 2. Read Bronze Data
df_bronze = spark.read.table(bronze_table)
source_count = df_bronze.count()
print(f"Source Count: {source_count}")

In [0]:
display(df_bronze)

In [0]:
# 3. Transform & Enforce Schema Rules
# - Cast strings to correct data types (DATE, INT)
# - Add current timestamp for _load_ts
# - Drop Bronze-only metadata (_ingest_ts, _source_file)
# - Keep _batch and _run_id unchanged (Carry Forward)
df_silver = df_bronze.withColumn("batchdate", col("batchdate").cast("date")) \
                     .withColumn("batchid", col("batchid").cast("int")) \
                     .withColumn("_load_ts", current_timestamp()) \
                     .drop("_ingest_ts", "_source_file")

In [0]:
# 4. Deduplicate (Keep the latest record per batchid)
# We partition by batchid and take the most recent record
window_spec = Window.partitionBy("batchid").orderBy(col("_load_ts").desc())
df_final = df_silver.withColumn("rn", row_number().over(window_spec)) \
                    .filter(col("rn") == 1) \
                    .drop("rn")

In [0]:
# 5. Overwrite Silver Table
# Lookups use OVERWRITE mode because they are small reference tables rebuilt each run
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.silver;")
df_final.write.format("delta").mode("overwrite").saveAsTable(silver_table)

target_count = spark.read.table(silver_table).count()
print(f"Success! Processed {target_count} rows into Silver.")

In [0]:
spark.sql(f"use catalog {catalog_name}")

In [0]:
# 6. Operations Logging
# Extract the _run_id from the dataframe to link the audit logs to the original pipeline run
carried_run_id = str(df_final.select("`_run_id`").first()[0])


log_pipeline_recon(
    spark=spark,
    run_id=carried_run_id,
    batch_id="ALL",
    domain="CONTROL",
    table_name="batchdate",
    source_layer="bronze",
    target_layer="silver",
    source_count=source_count,
    target_count=target_count
)

log_audit_event(
    spark=spark,
    run_id=carried_run_id,
    batch="ALL",
    layer="silver",
    table_name="batchdate",
    operation="OVERWRITE",
    rows_affected=target_count
)